In [ ]:
import json
import pathlib
import pandas as pd

# === 1. 一些配置 ===

BASE_DIR = pathlib.Path("../javascript/output/comm1")  # TODO: 改成你的目录
N_RANKS = 128                        # rank 数量

# 不同 dtype 的字节数，可以按实际补充
DTYPE_SIZE = {
    "Char": 1,
    "Byte": 1,
    "Float": 4,
    "Half": 2,
    "BFloat16": 2,
    "Double": 8,
}


def load_one_rank(rank: int) -> pd.DataFrame:
    """
    读取单个 rank 的 JSON 文件，提取 all_to_all 记录，并计算发送/接收字节数。
    """
    path = BASE_DIR / f"comm_rank{rank}_step4.json"
    if not path.exists():
        print(f"[WARN] file not found for rank {rank}: {path}")
        return pd.DataFrame()

    with path.open() as f:
        data = json.load(f)

    rows = []
    for item in data:
        # 结构：item["param"][0]["args"] / item["marker"]["span"]
        try:
            args = item["param"][0]["args"]
            span = item["marker"]["span"]
        except (KeyError, IndexError, TypeError):
            continue

        if args.get("Collective name") != "all_to_all":
            continue

        ts = span.get("ts", 0)

        # 元素个数
        in_nelems = args.get("In msg nelems", 0)
        out_nelems = args.get("Out msg nelems", 0)

        # dtype -> 单个元素字节数
        dtype = args.get("dtype", "Char")
        elem_size = DTYPE_SIZE.get(dtype, 1)

        # ====== 这里是关键：计算发送/接收字节数 ======
        send_bytes = out_nelems * elem_size   # 对外发送的数据量
        recv_bytes = in_nelems * elem_size    # 从外部接收的数据量
        total_bytes = send_bytes + recv_bytes # 如果你想看总通信量可以用这个

        rows.append({
            "rank": rank,
            "ts": ts,
            "send_bytes": send_bytes,
            "recv_bytes": recv_bytes,
            "total_bytes": total_bytes,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # 按 ts 排序，并在“本 rank 内”从 0 开始标记 all_to_all 的顺序
    df = df.sort_values("ts").reset_index(drop=True)
    df["ts_id"] = range(len(df))  # 0,1,2,...  每个 all_to_all 的离散化时间 id

    return df


# === 2. 收集所有 rank 的数据 ===

all_dfs = []
for r in range(N_RANKS):
    df_r = load_one_rank(r)
    if not df_r.empty:
        all_dfs.append(df_r)

df = pd.concat(all_dfs, ignore_index=True)
print(df.head())
print("shape:", df.shape)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# === 3. 把数据透视成 (rank, ts_id) 的矩阵 ===

# 这里使用 send_bytes，如果你想看 recv_bytes 或 total_bytes，改下面这一行的值即可
VALUE_COL = "send_bytes"  # <-- 改成 "recv_bytes" 或 "total_bytes" 就可以

pivot = df.pivot_table(
    index="rank",
    columns="ts_id",
    values=VALUE_COL,
    aggfunc="sum",
    fill_value=0,
)

print("pivot shape:", pivot.shape)

# === 4. 画热力图 ===

plt.figure(figsize=(12, 8))
img = plt.imshow(pivot.values, aspect="auto")  # 不设置 cmap，用默认即可
plt.colorbar(img, label=f"{VALUE_COL} (bytes)")
plt.xlabel("ts_id (discrete all_to_all index per rank)")
plt.ylabel("rank id")
plt.title(f"All-to-all {VALUE_COL} per rank per ts_id")

# y 轴加上 rank 数字（如果 128 行太密，可以只显示部分）
plt.yticks(np.arange(len(pivot.index)), pivot.index)

plt.tight_layout()
plt.show()
